# 4a Prepare Proposals For Analysis

This notebook prepares all proposal-side artifacts for downstream analyses.

For each requested condition it builds both the `original` and `rephrased` branches, then saves:
- canonical proposal master tables
- full-text and abstract embedding bundles
- pairwise cosine matrices
- proposal-space UMAP caches
- proposal-to-literature nearest-neighbor caches

It also prepares the shared literature corpus, embeddings, BERTopic region assignments, and literature-only UMAP assets once per run.

In [6]:
CONDITIONS_TO_PREPARE = ['baseline', 'one_at_a_time', 'persona']
TEXT_VERSIONS = ['original', 'rephrased']

EMBEDDING_MODEL_NAME = 'michiyasunaga/BioLinkBERT-large'
PROPOSAL_TO_LITERATURE_K = 50
REUSE_EXISTING_ARTIFACTS = True
WRITE_JSON_COMPANIONS = True


In [7]:
import json
from datetime import datetime
from pathlib import Path
import pandas as pd

import sys
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from prepare_proposals_for_analysis import (
    PROPOSAL_EMBEDDING_MODEL,
    PROPOSAL_TO_LITERATURE_K as DEFAULT_K,
    build_embedding_bundle,
    build_proposal_master_table,
    compute_or_load_proposal_to_literature_knn,
    compute_pairwise_cosine_matrix,
    find_project_root,
    fit_or_load_proposal_umap,
    load_ai_original_proposals,
    load_ai_rephrased_proposals,
    load_human_original_proposals,
    load_human_rephrased_proposals,
    locate_latest_proposal_files,
    validate_proposal_alignment,
    write_prepare_manifest,
)
from prepare_literature_assets import (
    build_or_load_literature_embeddings,
    compute_corpus_hash,
    fit_or_load_literature_bertopic,
    fit_or_load_literature_umap,
    load_literature_corpus,
    normalize_literature_articles,
    write_literature_manifest,
)

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
if EMBEDDING_MODEL_NAME != PROPOSAL_EMBEDDING_MODEL:
    print(f'Notebook embedding model override active: {EMBEDDING_MODEL_NAME}')
if PROPOSAL_TO_LITERATURE_K != DEFAULT_K:
    print(f'Notebook proposal-to-literature K override active: {PROPOSAL_TO_LITERATURE_K}')

LITERATURE_RAW_PATH = PROJECT_ROOT / 'data' / 'literature' / 'relevant-corpus-from-pubmed.json'
PREPARED_LITERATURE_DIR = PROJECT_ROOT / 'data' / 'prepared' / 'literature'
PREPARED_LITERATURE_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDINGS_LITERATURE_DIR = PROJECT_ROOT / 'data' / 'embeddings' / 'literature'
EMBEDDINGS_LITERATURE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Conditions to prepare: {CONDITIONS_TO_PREPARE}')
print(f'Text versions: {TEXT_VERSIONS}')


Project root: /Users/eveyhuang/Documents/NICO/human-AI-proposal
Conditions to prepare: ['baseline', 'one_at_a_time', 'persona']
Text versions: ['original', 'rephrased']


In [8]:
human_original_df = load_human_original_proposals(PROJECT_ROOT)
human_rephrased_df = load_human_rephrased_proposals(PROJECT_ROOT)
human_alignment_issues = validate_proposal_alignment(
    human_original_df.rename(columns={'proposal_title': 'title'}),
    human_rephrased_df.rename(columns={'proposal_title': 'title'}),
    'human',
)
if human_alignment_issues:
    raise RuntimeError('Human proposal alignment failed: ' + '; '.join(human_alignment_issues))

literature_payload = load_literature_corpus(LITERATURE_RAW_PATH)
literature_article_index = normalize_literature_articles(literature_payload)
literature_corpus_hash = compute_corpus_hash(literature_payload)

prepared_literature_json = PREPARED_LITERATURE_DIR / 'literature_corpus_prepared.json'
prepared_literature_csv = PREPARED_LITERATURE_DIR / 'literature_article_index.csv'
literature_manifest_path = PREPARED_LITERATURE_DIR / 'literature_prepare_manifest.json'

prepared_literature_json.write_text(json.dumps(literature_payload, indent=2, ensure_ascii=False))
literature_article_index.to_csv(prepared_literature_csv, index=False)

literature_embeddings_path = EMBEDDINGS_LITERATURE_DIR / 'relevant_literature_embeddings.pkl'
literature_bundle = build_or_load_literature_embeddings(
    literature_article_index,
    output_path=literature_embeddings_path,
    corpus_hash=literature_corpus_hash,
    model_name=EMBEDDING_MODEL_NAME,
    reuse_if_exists=REUSE_EXISTING_ARTIFACTS,
)

bertopic_model_path = EMBEDDINGS_LITERATURE_DIR / 'lit_bertopic_model.pkl'
bertopic_assignments_path = PREPARED_LITERATURE_DIR / 'lit_bertopic_assignments.csv'
bertopic_topic_info_path = PREPARED_LITERATURE_DIR / 'lit_bertopic_topic_info.csv'
lit_umap_reducer_path = EMBEDDINGS_LITERATURE_DIR / 'lit_umap_reducer.pkl'
lit_umap_coords_path = EMBEDDINGS_LITERATURE_DIR / 'lit_umap2d.npy'

bertopic_status = 'not_run'
bertopic_error = ''
try:
    bertopic_model, bertopic_assignments_df, bertopic_topic_info_df = fit_or_load_literature_bertopic(
        literature_article_index,
        literature_bundle,
        model_path=bertopic_model_path,
        assignments_path=bertopic_assignments_path,
        topic_info_path=bertopic_topic_info_path,
        reuse_if_exists=REUSE_EXISTING_ARTIFACTS,
    )
    bertopic_status = 'ready'
except Exception as exc:
    bertopic_assignments_df = pd.DataFrame()
    bertopic_topic_info_df = pd.DataFrame()
    bertopic_status = 'failed'
    bertopic_error = str(exc)
    print(f'BERTopic step failed: {exc}')

lit_umap_status = 'not_run'
lit_umap_error = ''
try:
    lit_reducer, lit_umap2d = fit_or_load_literature_umap(
        literature_bundle,
        reducer_path=lit_umap_reducer_path,
        coords_path=lit_umap_coords_path,
        reuse_if_exists=REUSE_EXISTING_ARTIFACTS,
    )
    lit_umap_status = 'ready'
except Exception as exc:
    lit_umap2d = None
    lit_umap_status = 'failed'
    lit_umap_error = str(exc)
    print(f'Literature UMAP step failed: {exc}')

literature_manifest = {
    'prepared_at': datetime.now().isoformat(),
    'source_file': str(LITERATURE_RAW_PATH),
    'prepared_corpus_json': str(prepared_literature_json),
    'prepared_article_index_csv': str(prepared_literature_csv),
    'embedding_file': str(literature_embeddings_path),
    'embedding_model_name': EMBEDDING_MODEL_NAME,
    'corpus_hash': literature_corpus_hash,
    'article_count': int(len(literature_article_index)),
    'bertopic_status': bertopic_status,
    'bertopic_model_file': str(bertopic_model_path),
    'bertopic_assignments_file': str(bertopic_assignments_path),
    'bertopic_topic_info_file': str(bertopic_topic_info_path),
    'bertopic_error': bertopic_error,
    'literature_umap_status': lit_umap_status,
    'literature_umap_reducer_file': str(lit_umap_reducer_path),
    'literature_umap_coords_file': str(lit_umap_coords_path),
    'literature_umap_error': lit_umap_error,
}
write_literature_manifest(literature_manifest_path, literature_manifest)
print(f'Human proposals: {len(human_original_df)} original / {len(human_rephrased_df)} rephrased')
print(f'Literature articles: {len(literature_article_index)}')
print(f'BERTopic status: {bertopic_status}')
print(f'Literature UMAP status: {lit_umap_status}')


2026-07-10 08:23:07,520 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-10 08:23:34,317 - BERTopic - Dimensionality - Completed ✓
2026-07-10 08:23:34,318 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-10 08:23:34,407 - BERTopic - Cluster - Completed ✓
2026-07-10 08:23:34,411 - BERTopic - Representation - Fine-tuning topics using representation models.


BERTopic step failed: max_df corresponds to < documents than min_df
Human proposals: 23 original / 23 rephrased
Literature articles: 39538
BERTopic status: failed
Literature UMAP status: ready


In [9]:
condition_prepare_outputs = {}

for condition in CONDITIONS_TO_PREPARE:
    print(f'\n=== Prepare proposals: {condition} ===')
    proposal_files = locate_latest_proposal_files(PROJECT_ROOT, condition)
    ai_original_df = load_ai_original_proposals(proposal_files['original'], condition)
    ai_rephrased_df = load_ai_rephrased_proposals(proposal_files['rephrased'], condition)
    ai_alignment_issues = validate_proposal_alignment(ai_original_df, ai_rephrased_df, f'ai::{condition}')
    if ai_alignment_issues:
        raise RuntimeError('AI proposal alignment failed: ' + '; '.join(ai_alignment_issues))

    if ai_original_df['proposal_uid'].nunique() != len(ai_original_df):
        raise RuntimeError(f'{condition}: duplicate AI proposal_uid values in original proposals')
    if len(ai_original_df) != 69:
        print(f'WARNING: {condition} has {len(ai_original_df)} AI proposals; expected 69 after redesign.')

    condition_outputs = {}
    for text_version in TEXT_VERSIONS:
        output_dir = PROJECT_ROOT / 'data' / 'prepared' / condition / 'proposals' / text_version
        output_dir.mkdir(parents=True, exist_ok=True)

        master_df = build_proposal_master_table(
            condition=condition,
            text_version=text_version,
            ai_original_df=ai_original_df,
            ai_rephrased_df=ai_rephrased_df,
            human_original_df=human_original_df,
            human_rephrased_df=human_rephrased_df,
        )

        master_csv_path = output_dir / 'proposal_master.csv'
        master_json_path = output_dir / 'proposal_master.json'
        embeddings_full_path = output_dir / 'proposal_embeddings_full.pkl'
        embeddings_abstract_path = output_dir / 'proposal_embeddings_abstract.pkl'
        pairwise_full_path = output_dir / 'proposal_pairwise_cosine_full.npy'
        pairwise_abstract_path = output_dir / 'proposal_pairwise_cosine_abstract.npy'
        proposal_umap_reducer_path = output_dir / 'proposal_umap_reducer.pkl'
        proposal_umap_coords_path = output_dir / 'proposal_umap2d.npy'
        proposal_knn_path = output_dir / 'proposal_to_literature_knn.npz'
        manifest_path = output_dir / 'prepare_manifest.json'

        master_df.to_csv(master_csv_path, index=False)
        if WRITE_JSON_COMPANIONS:
            master_json_path.write_text(json.dumps(master_df.to_dict('records'), indent=2, ensure_ascii=False))

        full_bundle = build_embedding_bundle(
            master_df,
            text_field='full_text',
            output_path=embeddings_full_path,
            model_name=EMBEDDING_MODEL_NAME,
            pooling='cls',
            reuse_if_exists=REUSE_EXISTING_ARTIFACTS,
        )
        abstract_bundle = build_embedding_bundle(
            master_df,
            text_field='abstract_text',
            output_path=embeddings_abstract_path,
            model_name=EMBEDDING_MODEL_NAME,
            pooling='cls',
            reuse_if_exists=REUSE_EXISTING_ARTIFACTS,
        )
        full_pairwise = compute_pairwise_cosine_matrix(full_bundle, pairwise_full_path)
        abstract_pairwise = compute_pairwise_cosine_matrix(abstract_bundle, pairwise_abstract_path)
        proposal_reducer, proposal_umap2d = fit_or_load_proposal_umap(
            full_bundle,
            reducer_path=proposal_umap_reducer_path,
            coords_path=proposal_umap_coords_path,
            reuse_if_exists=REUSE_EXISTING_ARTIFACTS,
        )
        proposal_knn_payload = compute_or_load_proposal_to_literature_knn(
            abstract_bundle,
            literature_bundle,
            output_path=proposal_knn_path,
            k=PROPOSAL_TO_LITERATURE_K,
            reuse_if_exists=REUSE_EXISTING_ARTIFACTS,
        )

        manifest = {
            'prepared_at': datetime.now().isoformat(),
            'condition': condition,
            'text_version': text_version,
            'input_original_ai_file': str(proposal_files['original']),
            'input_rephrased_ai_file': str(proposal_files['rephrased']),
            'human_original_rows': int(len(human_original_df)),
            'human_rephrased_rows': int(len(human_rephrased_df)),
            'ai_original_rows': int(len(ai_original_df)),
            'ai_rephrased_rows': int(len(ai_rephrased_df)),
            'proposal_master_rows': int(len(master_df)),
            'proposal_uid_order': master_df['proposal_uid'].astype(str).tolist(),
            'embedding_model_name': EMBEDDING_MODEL_NAME,
            'full_embedding_file': str(embeddings_full_path),
            'abstract_embedding_file': str(embeddings_abstract_path),
            'pairwise_full_file': str(pairwise_full_path),
            'pairwise_abstract_file': str(pairwise_abstract_path),
            'proposal_umap_reducer_file': str(proposal_umap_reducer_path),
            'proposal_umap_coords_file': str(proposal_umap_coords_path),
            'proposal_to_literature_knn_file': str(proposal_knn_path),
            'proposal_to_literature_k': PROPOSAL_TO_LITERATURE_K,
            'proposal_to_literature_built_from': 'abstract_text',
            'literature_corpus_hash': literature_corpus_hash,
            'pairwise_full_shape': list(full_pairwise.shape),
            'pairwise_abstract_shape': list(abstract_pairwise.shape),
            'proposal_umap_shape': list(proposal_umap2d.shape),
            'knn_rows': int(len(proposal_knn_payload['proposal_uids'])),
            'persona_nonnull_count': int(master_df['persona_card_id'].fillna('').astype(str).str.strip().ne('').sum()) if 'persona_card_id' in master_df.columns else 0,
        }
        write_prepare_manifest(manifest_path, manifest)

        condition_outputs[text_version] = {
            'master_df': master_df,
            'master_csv_path': master_csv_path,
            'manifest_path': manifest_path,
            'full_embedding_path': embeddings_full_path,
            'abstract_embedding_path': embeddings_abstract_path,
            'pairwise_full_path': pairwise_full_path,
            'pairwise_abstract_path': pairwise_abstract_path,
            'proposal_umap_path': proposal_umap_coords_path,
            'proposal_knn_path': proposal_knn_path,
        }
        print(f'  {text_version}: saved {len(master_df)} proposals -> {master_csv_path}')

    condition_prepare_outputs[condition] = condition_outputs



=== Prepare proposals: baseline ===
  WARNING ai::baseline: title normalization drift on 69/69 matched proposals (pairing is by proposal_uid, not title).
  WARNING ai::baseline: title normalization drift on 69/69 matched proposals (pairing is by proposal_uid, not title).
  original: saved 92 proposals -> /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/baseline/proposals/original/proposal_master.csv
  WARNING ai::baseline: title normalization drift on 69/69 matched proposals (pairing is by proposal_uid, not title).


Embedding texts: 100%|██████████| 12/12 [00:06<00:00,  1.96it/s]


  rephrased: saved 92 proposals -> /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/baseline/proposals/rephrased/proposal_master.csv

=== Prepare proposals: one_at_a_time ===


Embedding texts: 100%|██████████| 12/12 [00:13<00:00,  1.14s/it]


  original: saved 92 proposals -> /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/one_at_a_time/proposals/original/proposal_master.csv


...

  rephrased: saved 92 proposals -> /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/one_at_a_time/proposals/rephrased/proposal_master.csv

=== Prepare proposals: persona ===


Embedding texts: 100%|██████████| 12/12 [00:14<00:00,  1.17s/it]


  original: saved 92 proposals -> /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/persona/proposals/original/proposal_master.csv


Embedding texts: 100%|██████████| 12/12 [00:07<00:00,  1.65it/s]


  rephrased: saved 92 proposals -> /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/prepared/persona/proposals/rephrased/proposal_master.csv


In [10]:
summary_rows = []
for condition, branch_outputs in condition_prepare_outputs.items():
    for text_version, result in branch_outputs.items():
        master_df = result['master_df']
        summary_rows.append({
            'condition': condition,
            'text_version': text_version,
            'proposal_rows': len(master_df),
            'human_rows': int((master_df['source_type'] == 'human').sum()),
            'ai_rows': int((master_df['source_type'] == 'ai').sum()),
            'master_csv': str(result['master_csv_path']),
            'manifest': str(result['manifest_path']),
            'full_embeddings': str(result['full_embedding_path']),
            'abstract_embeddings': str(result['abstract_embedding_path']),
            'pairwise_full': str(result['pairwise_full_path']),
            'proposal_umap': str(result['proposal_umap_path']),
            'proposal_knn': str(result['proposal_knn_path']),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df


,condition,text_version,proposal_rows,human_rows,ai_rows,master_csv,manifest,full_embeddings,abstract_embeddings,pairwise_full,proposal_umap,proposal_knn
0,baseline,original,92,23,69,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
1,baseline,rephrased,92,23,69,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
2,one_at_a_time,original,92,23,69,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
3,one_at_a_time,rephrased,92,23,69,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
4,persona,original,92,23,69,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
5,persona,rephrased,92,23,69,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
